In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]

# Remove the bad env var if you set it earlier
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, date_format,to_date,
    hour, dayofweek, dayofmonth, month, year,
    when, split, get_json_object
)
from pyspark.sql.types import IntegerType, BooleanType

load_dotenv()

True

In [2]:
GCS_BUCKET = os.getenv('GCS_BUCKET_NAME')
GCP_PROJECT = os.getenv('GCP_PROJECT_ID')
BIGQUERY_DATASET = os.getenv('BIGQUERY_DATASET')

In [3]:
def create_spark_session():
    #print(f"⚙️  Creating Spark session with {shuffle_partitions} shuffle partitions")

    # Path to application default credentials
    credentials_path = os.path.expanduser(
        "~/.config/gcloud/application_default_credentials.json"
    )
    #print(f"🔑 Using credentials: {credentials_path}")

    java_opts = (
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.nio=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED "
    "--add-opens java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens java.base/sun.security.action=ALL-UNNAMED"
    )

    return (SparkSession.builder
        .appName("GitHubArchiveTransformation")
        .config("spark.driver.extraJavaOptions", java_opts)
        .config("spark.executor.extraJavaOptions", java_opts)

        # GCS connector
        .config("spark.jars",
                "/home/codespace/gcs-connector-hadoop3-latest.jar,"
                "/home/codespace/spark-bigquery-with-dependencies_2.12-0.36.1.jar")
        .config("spark.hadoop.fs.gs.impl",
                "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
        .config("spark.hadoop.fs.AbstractFileSystem.gs.impl",
                "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")

        # Use JSON key file for authentication
        .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
        .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile",
                credentials_path)

        # Performance
        #.config("spark.sql.shuffle.partitions", str(shuffle_partitions))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.adaptive.skewJoin.enabled", "true")

        .getOrCreate()
    )


In [ ]:
def read_raw_data(spark, date_str):
    """Read all hourly files for a given date from GCS"""
    # Wildcard picks up all 24 hourly files: 2025-01-01-0 to 2025-01-01-23
    gcs_path = f"gs://{GCS_BUCKET}/raw/github/{date_str}/{date_str}-*.json.gz"
    print(f"\n📖 Reading all hourly files for {date_str}...")
    print(f"   Path: {gcs_path}")

    df = spark.read.json(gcs_path)

    #print(f"📊 Total rows: {df.count():,}") # ← this forces a full data scan
    print(f"📊 Execution partitions after reading: {df.rdd.getNumPartitions()}")

    return df

In [ ]:
spark = create_spark_session()
spark.sparkContext.setLogLevel("ERROR")  # ← suppress logs immediately


26/03/30 23:12:40 WARN Utils: Your hostname, codespaces-c39ba7 resolves to a loopback address: 127.0.0.1; using 10.0.0.252 instead (on interface eth0)
26/03/30 23:12:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/30 23:12:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [24]:
date_str = "2025-01-03"
df = read_raw_data(spark, date_str)


📖 Reading all hourly files for 2025-01-03...
   Path: gs://reddit-analytic-pipeline-bucket/raw/github/2025-01-03/2025-01-03-*.json.gz


📊 Total rows: 4,740,429
📊 Execution partitions after reading: 24


In [ ]:
# event_types = ["PushEvent", "WatchEvent", "PullRequestEvent", "IssuesEvent", "ForkEvent"]

# for event_type in event_types:
#     print(f"\n📊 {event_type} payload fields:")
#     df.filter(df.type == event_type) \
#     .select("payload.*") \
#     .show(2, truncate=True)


📊 PushEvent payload fields:


+------+--------------------+-------+--------------------+-----------+-------------+------+--------------------+-----+-------------+------+------+-----+------------+-----------+-----------+-----------------+--------+-------+-------------+------+----+
|action|              before|comment|             commits|description|distinct_size|forkee|                head|issue|master_branch|member|number|pages|pull_request|    push_id|pusher_type|              ref|ref_type|release|repository_id|review|size|
+------+--------------------+-------+--------------------+-----------+-------------+------+--------------------+-----+-------------+------+------+-----+------------+-----------+-----------+-----------------+--------+-------+-------------+------+----+
|  NULL|1c0bb7ae07c5755de...|   NULL|[{{41898282+githu...|       NULL|            1|  NULL|9e2f17a85a5ca0117...| NULL|         NULL|  NULL|  NULL| NULL|        NULL|21928742294|       NULL|refs/heads/output|    NULL|   NULL|    314538798|  NULL|  

+-------+------+-------+-------+-----------+-------------+------+----+-----+-------------+------+------+-----+------------+-------+-----------+----+--------+-------+-------------+------+----+
| action|before|comment|commits|description|distinct_size|forkee|head|issue|master_branch|member|number|pages|pull_request|push_id|pusher_type| ref|ref_type|release|repository_id|review|size|
+-------+------+-------+-------+-----------+-------------+------+----+-----+-------------+------+------+-----+------------+-------+-----------+----+--------+-------+-------------+------+----+
|started|  NULL|   NULL|   NULL|       NULL|         NULL|  NULL|NULL| NULL|         NULL|  NULL|  NULL| NULL|        NULL|   NULL|       NULL|NULL|    NULL|   NULL|         NULL|  NULL|NULL|
|started|  NULL|   NULL|   NULL|       NULL|         NULL|  NULL|NULL| NULL|         NULL|  NULL|  NULL| NULL|        NULL|   NULL|       NULL|NULL|    NULL|   NULL|         NULL|  NULL|NULL|
+-------+------+-------+-------+--------

In [ ]:
# # PullRequest fields
# print("\n📊 PullRequest fields:")
# df.filter(df.type == "PullRequestEvent") \
#     .select("payload.pull_request.*") \
#     .show(2, truncate=True)

# # Issue fields
# print("\n📊 Issue fields:")
# df.filter(df.type == "IssuesEvent") \
#     .select("payload.issue.*") \
#     .show(2, truncate=True)

# # Forkee fields
# print("\n📊 Forkee fields:")
# df.filter(df.type == "ForkEvent") \
#     .select("payload.forkee.*") \
#     .show(2, truncate=True)


📊 PullRequest fields:


+--------------------+------------------+---------+--------+---------+------------------+----------+--------------------+--------------------+-------------+---------+--------+--------------------+-------+--------------------+--------------------+---------+--------------------+-----+--------------------+--------------------+----------+--------------------+------+------+---------------------+--------------------+---------+---------------+------+---------+---------+---------+-------------------+------+--------------------+----------+-------------------+---------------+--------------------+---------------+--------------------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|              _links|active_lock_reason|additions|assignee|assignees|author_association|auto_merge|                base|                body|changed_files|closed_at|comments|        comments_url|commits|         commits_url|          created_at|deletions|  

In [20]:
def transform_events(df):
    """Extract and transform relevant fields"""
    print("\n🔄 Transforming events...")

    transformed = df.select(
        # Event info
        col("id").alias("event_id"),
        col("type").alias("event_type"),
        to_timestamp(col("created_at")).alias("created_at"),

        # Date dimensions
        to_date(col("created_at")).alias("event_date"),
        year(col("created_at")).alias("event_year"),
        month(col("created_at")).alias("event_month"),
        dayofmonth(col("created_at")).alias("event_day"),
        hour(col("created_at")).alias("event_hour"),
        dayofweek(col("created_at")).alias("day_of_week"),

        # Weekend flag
        when(
            dayofweek(col("created_at")).isin([1, 7]), True
        ).otherwise(False).alias("is_weekend"),

        # Public flag
        col("public").alias("is_public"),

        # Actor info
        col("actor.id").alias("actor_id"),
        col("actor.login").alias("actor_login"),

        # Org info
        col("org.login").alias("org_login"),
        col("org.id").alias("org_id"),

        # Repo info
        col("repo.id").alias("repo_id"),
        col("repo.name").alias("repo_name"),
        split(col("repo.name"), "/")[0].alias("repo_owner"),
        split(col("repo.name"), "/")[1].alias("repo_short_name"),

        # PushEvent fields
        col("payload.size").cast("integer").alias("push_size"),
        col("payload.distinct_size").cast("integer").alias("push_distinct_size"),
        col("payload.ref").alias("push_ref"),
        col("payload.repository_id").alias("push_repository_id"),

        # WatchEvent
        col("payload.action").alias("action"),

        # PullRequestEvent fields
        col("payload.number").alias("pr_number"),
        col("payload.pull_request.merged").alias("pr_merged"),
        col("payload.pull_request.state").alias("pr_state"),
        col("payload.pull_request.additions").cast("integer").alias("pr_additions"),
        col("payload.pull_request.deletions").cast("integer").alias("pr_deletions"),
        col("payload.pull_request.changed_files").cast("integer").alias("pr_changed_files"),
        col("payload.pull_request.base.repo.language").alias("repo_language"),
        col("payload.pull_request.title").alias("pr_title"),

        # IssuesEvent fields
        col("payload.issue.state").alias("issue_state"),
        col("payload.issue.number").alias("issue_number"),
        col("payload.issue.title").alias("issue_title"),
        col("payload.issue.comments").cast("integer").alias("issue_comments"),

        # ForkEvent fields
        col("payload.forkee.full_name").alias("forkee_name"),
        col("payload.forkee.language").alias("forkee_language"),
        col("payload.forkee.stargazers_count").cast("integer").alias("forkee_stars"),
        col("payload.forkee.forks_count").cast("integer").alias("forkee_forks"),
        col("payload.forkee.description").alias("forkee_description"),
    )

    return transformed

In [8]:
def filter_public_events(df):
    """Keep only public events"""
    filtered = df.filter(col("is_public") == True)
    return filtered

In [9]:
def show_statistics(df):
    """Show useful statistics"""
    total = df.count()
    print(f"\n📊 Total events: {total:,}")
    print(f"📊 Execution partitions: {df.rdd.getNumPartitions()}")

    print("\n📊 Event type distribution:")
    df.groupBy("event_type") \
        .count() \
        .orderBy("count", ascending=False) \
        .show()

    print("\n📊 Sample data:")
    df.show(10, truncate=False)

In [25]:
df_transformed = transform_events(df)

# Filter public only
df_filtered = filter_public_events(df_transformed)

# Show schema
#print("\n📊 Transformed Schema:")
#df_filtered.printSchema()

# # Show statistics
# show_statistics(df_filtered)



🔄 Transforming events...


In [12]:
def save_to_gcs_parquet(df, date_str):
    """
    Save transformed data as parquet to GCS for a full day.
    Partitioned by event_date for fast reads later.
    """

    output_path = f"gs://{GCS_BUCKET}/processed/github/{date_str}"
    print(f"\n💾 Saving parquet to: {output_path}")

    df.write \
        .mode("overwrite") \
        .partitionBy("event_date") \
        .parquet(output_path)

    print(f"✅ Saved to: {output_path}")



In [26]:
save_to_gcs_parquet(df_filtered, "2025-01-03")


💾 Saving parquet to: gs://reddit-analytic-pipeline-bucket/processed/github/2025-01-03


✅ Saved to: gs://reddit-analytic-pipeline-bucket/processed/github/2025-01-03


In [31]:
def save_to_bigquery(table_name, date_str):
    full_table = f"{GCP_PROJECT}.{BIGQUERY_DATASET}.{table_name}"
    parquet_path = f"gs://{GCS_BUCKET}/processed/github/{date_str}"
    print(f"\n💾 Loading to BigQuery: {full_table}")
    print(f"   Source: {parquet_path}")

    # Delete existing partition for this date before writing
    from google.cloud import bigquery
    from google.cloud.exceptions import NotFound

    client = bigquery.Client(project=GCP_PROJECT)
    
    # Only delete partition if table already exists
    try:
        client.get_table(full_table)  # check if table exists
        query = f"""
            DELETE FROM `{full_table}`
            WHERE event_date = '{date_str}'
        """
        client.query(query).result()
        print(f"🗑️  Cleared partition: {date_str}")
    except NotFound:
        print(f"📋 Table does not exist yet, will be created on write")

    df_parquet = spark.read.parquet(parquet_path)

    df_parquet.write \
        .format("bigquery") \
        .option("table", full_table) \
        .option("temporaryGcsBucket", GCS_BUCKET) \
        .option("partitionField", "event_date") \
        .option("clusteredFields", "event_type") \
        .option("allowFieldAddition", "true") \
        .option("allowFieldRelaxation", "true") \
        .mode("append") \
        .save()

    print(f"✅ Loaded to BigQuery: {full_table}")

In [32]:
save_to_bigquery("github_events", "2025-01-03")


💾 Loading to BigQuery: reddit-analytic-pipeline.reddit_analytics.github_events
   Source: gs://reddit-analytic-pipeline-bucket/processed/github/2025-01-03
📋 Table does not exist yet, will be created on write


✅ Loaded to BigQuery: reddit-analytic-pipeline.reddit_analytics.github_events


In [ ]:
spark.stop()